In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic' 
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
df = pd.read_csv('../../data/DieCasting_Quality_Raw_Data.csv',header=[0,1])
df

1. 컬럼명 공백제거
2. 중복제거
3. 2번, 3번을 불량(1)로 대체
4. 새로운 파생컬럼 생성

In [ ]:
### 1. 컬럼명 공백 제거
df.rename(columns={'Biscuit_Thickness ': 'Biscuit_Thickness', 'Clamping_Force ': 'Clamping_Force', ' Pressure_Rise_Time': 'Pressure_Rise_Time'}, level=1, inplace=True)

In [ ]:
### 2. 중복 제거
# 제외할 컬럼 리스트 정의
exclude_cols = [('Process', 'id')]

# 전체 컬럼에서 제외할 컬럼만 뺀 리스트 생성
target_cols = df.columns.difference(exclude_cols).tolist()

# 중복 제거 실행 (첫 번째 행은 남기고 나머지는 삭제)
df_cleaned = df.drop_duplicates(subset=target_cols, keep='first').reset_index(drop=True)

# 결과 확인
print(f"제거 전 행 개수: {len(df)}")
print(f"제거 후 행 개수: {len(df_cleaned)}")
print(f"삭제된 중복 행 개수: {len(df) - len(df_cleaned)}")

In [ ]:
### 3. 2,3을 불량 1로 대체
defect_cols = df_cleaned['Defects'].columns

# 'Defects' 대분류 아래의 모든 컬럼에 대해 2 이상의 값을 1로 변경
for col in defect_cols:
    # Defects 하위의 col 값 중 2 이상인 경우를 1로 치환
    df_cleaned.loc[df_cleaned[('Defects', col)] >= 2, ('Defects', col)] = 1

In [ ]:
### 4. 새로운 파생컬럼 생성
defect_cols = [col for col in df_cleaned.columns if col[0]=='Defects']

df_cleaned[('Defect_Flag','Is_Defect')] = (df_cleaned[defect_cols] == 1).any(axis=1).astype(int)

df_cleaned[('Defect_Flag','Is_Defect')].value_counts() # --> 0: 정상, 1: 불량

In [ ]:
df_cleaned

In [ ]:
### 5. 모든 컬럼명 소문자화
df_cleaned.columns = df_cleaned.columns.map(lambda x: tuple(str(level).lower() for level in x))

In [ ]:
df_cleaned

In [ ]:
df_type_1 = df_cleaned[df_cleaned[('process', 'product_type')] == 1].reset_index(drop=True)

In [ ]:
df_type_1

In [ ]:
defect_cols = [col for col in df_type_1.columns if 'defect' in str(col)]

for col in defect_cols:
    print(f"--- [Column: {col}] ---")
    print(df_type_1[col].value_counts())
    print("\n") 

# 각 컬럼의 value_counts를 데이터프레임 하나로 병합
counts_summary = df_type_1[defect_cols].apply(lambda x: x.value_counts()).fillna(0).astype(int)
display(counts_summary)

In [ ]:
# 값이 모두 0인 컬럼명 리스트 생성
only_zero_cols = [col for col in defect_cols if (df_type_1[col] == 0).all()]

print(f"값이 0만 있는 컬럼들({len(only_zero_cols)}개): \n{only_zero_cols} ")

In [ ]:
d_reduced = df_type_1.drop(columns=only_zero_cols)

d_reduced

In [ ]:
# Product_Type이 1인 행만 필터링하여 저장
d_reduced.to_csv('product_type_1.csv', index=False, encoding='utf-8-sig')

In [ ]:
df_type_2 = df_cleaned[df_cleaned[('process', 'product_type')] == 2].reset_index(drop=True)

In [ ]:
defect_cols = [col for col in df_type_2.columns if 'defect' in str(col)]

for col in defect_cols:
    print(f"--- [Column: {col}] ---")
    print(df_type_2[col].value_counts())
    print("\n") 

# 각 컬럼의 value_counts를 데이터프레임 하나로 병합
counts_summary = df_type_2[defect_cols].apply(lambda x: x.value_counts()).fillna(0).astype(int)
display(counts_summary)

In [ ]:
# 값이 모두 0인 컬럼명 리스트 생성
only_zero_cols = [col for col in defect_cols if (df_type_2[col] == 0).all()]

print(f"값이 0만 있는 컬럼들({len(only_zero_cols)}개): \n{only_zero_cols} ")

In [ ]:
d_reduced = df_type_2.drop(columns=only_zero_cols)

d_reduced

In [ ]:
# 2. Product_Type이 2인 행만 필터링하여 저장
df_type_2.to_csv('product_type_2.csv', index=False, encoding='utf-8-sig')

In [ ]:
df1 = pd.read_csv('product_type_1.csv',header=[0,1])
df2 = pd.read_csv('product_type_2.csv',header=[0,1])

In [ ]:
# 컬럼별 결측치 개수 합계
print(df1.isnull().sum())

In [ ]:
df1

In [ ]:
# 컬럼별 결측치 개수 합계
print(df2.isnull().sum())

In [ ]:
df1.info()

In [ ]:
df2.info()